In [ ]:
import numpy as np
from pathlib import Path

In [ ]:
data = np.load('L3_Manifold.npy', allow_pickle=True)
obj = data.item() 
x1 = obj['x1']
v2 = obj['v2']
T = obj['T']
C = obj['C']
mu = obj['mu']
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

In [ ]:
np.min(x1)

In [ ]:
np.max(x1)

In [ ]:
!pip install -U pysr

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen

In [ ]:
plt.plot(x1, v2)
plt.grid()
plt.xlabel('x1')
plt.ylabel('v2')
#plt.legend()
plt.show()

In [ ]:
plt.plot(x1, T)
plt.grid()
plt.xlabel('x1')
plt.ylabel('T')
#plt.legend()
plt.show()

In [ ]:
model = PySRRegressor(
        niterations=5000,  # < Increase me for better results
        binary_operators=["*", "+", "-","/","^"],
        unary_operators=[
            'exp',
            'cos',
            'sin',
            'log',
        ],
        #extra_sympy_mappings={"pimult": lambda x: np.pi*x},
        #nested_constraints={"cos": {"cos": 0, "sin": 0},"sin": {"cos": 0, "sin": 0}},
        constraints={'^': (-1, 1)},
        #nested_constraints={"exp": {"exp": 0}},
        #nested_constraints={"exp": {"exp": 0},"log": {"log": 0}},
        nested_constraints={"exp": {"exp": 0}, "^": {"^": 0},"log": {"log": 0},"cos":{"cos": 0,"sin": 0, "exp": 0,"log": 0},"sin":{"cos": 0,"sin": 0,"exp": 0,"log": 0}},
        populations=36,
        verbosity=0,
        #batching=True,
        warm_start = False,
        #denoise=True
    )

In [ ]:
input_values = np.expand_dims(x1.flatten(),-1)
target_values = np.expand_dims(v2.flatten(),-1)
model.fit(input_values, target_values,
        variable_names=["x"])

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import x

In [ ]:
N=None

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
f = lambdastr(x, s)
print(f)
plt.plot(input_values, test_trues)
plt.plot(input_values, test_preds,'--')
plt.grid()
plt.show()

In [ ]:
mu

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
import math

def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt


x1_fun = lambda x: x
x2_fun = lambda x: 0

def v1_fun(x):
    return 0
def v2_fun(x):
    y = - (x + 1.0047855) * (1.8641998 - 0.10354004*math.cos(x**2)/x + 0.05944614206047/x**2 + 0.022025653372038/(x**2*math.cos(x + 0.46803632))) #4.5e-4
    return y
def T_fun(x):
    y = (math.log(-x + 184.123052928413*math.exp(math.sin(x + 1.3517317 - 1.3393477/x)**(-0.020119384)) + math.exp(math.cos(x**2 -3.9773538))))#9.7e-5
    return y
x_range = np.linspace(np.min(x1), np.max(x1), 10)

fig_man, ax_man = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], 'k')
    #ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_man.plot(L3, 0, '*k')
ax_man.grid()
#ax_res.grid()

ax_man.set_xlabel('x, безразм.', fontsize=11)
ax_man.set_ylabel('y, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L3_orbits.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')


In [ ]:

x_range = np.linspace(np.min(x1), np.max(x1), 100)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L3_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:

a = x1
trues = v2
preds = [v2_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x$, безразм.')
ax.set_ylabel(r'$v_y$, безразм.')
ax.legend(loc='lower left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='upper right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L3_vy.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
a = x1
trues = T
preds = [T_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x$, безразм.')
ax.set_ylabel(r'$T$, безразм.')
ax.legend()

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L3_T.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "- (x + 1.0047855) * (1.8641998 - 0.10354004*math.cos(x**2)/x + 0.05944614206047/x**2 + 0.022025653372038/(x**2*math.cos(x + 0.46803632))) #4.5e-4"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=8, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(math.log(-x + 184.123052928413*math.exp(math.sin(x + 1.3517317 - 1.3393477/x)**(-0.020119384)) + math.exp(math.cos(x**2 -3.9773538))))#9.7e-5"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=3, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from fractions import Fraction
print(Fraction(0.020119384).limit_denominator(100))

In [ ]:
5/84